# Load PMax LLM Outcomes from GCS to BigQuery

Incrementally loads `all_llm_outcomes.csv` files stored under
`gs://pmax_placement_automation/runs/` into
`yorkville-warehouse.glin_google_ads.pmax_all_llm_outcomes`. Each run checks which GCS source files are already represented in BigQuery and
appends rows only from unseen files. 

The backfill runs start Aug. 7 since that is when the PMax Automation job started.

In [ ]:
%pip install gcsfs google-cloud-bigquery pyarrow

In [ ]:
import json
import re

import gcsfs
import pandas as pd
from google.api_core.exceptions import NotFound
from google.cloud import bigquery
from google.oauth2 import service_account

dbutils.widgets.text("start_date", "2026-08-07", "Include runs starting on")

PROJECT_ID = "yorkville-warehouse"
DATASET_ID = "glin_google_ads"
TABLE_NAME = "pmax_all_llm_outcomes"
TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
BQ_LOCATION = "northamerica-northeast2"
RUNS_ROOT = "pmax_placement_automation/runs"

START_DATE_TEXT = dbutils.widgets.get("start_date").strip()

try:
    START_DATE = pd.Timestamp(START_DATE_TEXT, tz="UTC")
except ValueError as exc:
    raise ValueError("start_date must use YYYY-MM-DD format") from exc

print(f"Including runs at or after: {START_DATE.isoformat()}")
print(f"BigQuery destination: {TABLE_ID}")

In [ ]:
service_account_key = json.loads(
    dbutils.secrets.get(scope="yorkville_warehouse", key="key")
)
credentials = service_account.Credentials.from_service_account_info(
    service_account_key
)

fs = gcsfs.GCSFileSystem(
    project=PROJECT_ID,
    token=service_account_key,
)
bq_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials,
    location=BQ_LOCATION,
)

TABLE_SCHEMA = [
    bigquery.SchemaField("source_record_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("source_run_id", "STRING"),
    bigquery.SchemaField("source_run_date", "DATE"),
    bigquery.SchemaField("source_run_hour", "INTEGER"),
    bigquery.SchemaField("source_run_timestamp_utc", "TIMESTAMP"),
    bigquery.SchemaField("source_gcs_path", "STRING"),
    bigquery.SchemaField("source_row_number", "INTEGER"),
    bigquery.SchemaField("display_url", "STRING"),
    bigquery.SchemaField("assessment_type", "STRING"),
    bigquery.SchemaField("reassessment_reason", "STRING"),
    bigquery.SchemaField("current_impressions", "INTEGER"),
    bigquery.SchemaField("verdict", "STRING"),
    bigquery.SchemaField("confidence", "INTEGER"),
    bigquery.SchemaField("summary", "STRING"),
    bigquery.SchemaField("input_tokens", "INTEGER"),
    bigquery.SchemaField("output_tokens", "INTEGER"),
    bigquery.SchemaField("total_tokens", "INTEGER"),
    bigquery.SchemaField("reasoning_tokens", "INTEGER"),
    bigquery.SchemaField("scrape_failed", "BOOLEAN"),
    bigquery.SchemaField("loaded_at_utc", "TIMESTAMP"),
]

try:
    bq_client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
except NotFound as exc:
    raise RuntimeError(
        f"Create the {PROJECT_ID}.{DATASET_ID} dataset in {BQ_LOCATION} "
        "and grant the Databricks service account BigQuery Data Editor access"
    ) from exc

try:
    destination_table = bq_client.get_table(TABLE_ID)
    print(f"Using existing BigQuery table with {destination_table.num_rows:,} rows")
except NotFound:
    destination_table = bigquery.Table(TABLE_ID, schema=TABLE_SCHEMA)
    destination_table.time_partitioning = bigquery.TimePartitioning(
        type_=bigquery.TimePartitioningType.DAY,
        field="source_run_date",
    )
    destination_table.clustering_fields = ["verdict", "assessment_type"]
    destination_table = bq_client.create_table(destination_table)
    print(f"Created {TABLE_ID}")

print(
    "Authenticated to GCS and BigQuery as "
    f"{service_account_key['client_email']}"
)

In [ ]:
RUN_PATTERN = re.compile(
    r"runs/(?P<run_id>\d{4}-\d{2}-\d{2}(?:-\d{1,2})?)/"
    r"(?:pmax_)?all_llm_outcomes\.csv$",
    re.IGNORECASE,
)


def parse_run(path):
    match = RUN_PATTERN.search(path)
    if not match:
        return None

    run_id = match.group("run_id")
    run_date = run_id[:10]
    run_hour = int(run_id[11:]) if len(run_id) > 10 else 0
    run_timestamp = pd.Timestamp(run_date, tz="UTC") + pd.Timedelta(hours=run_hour)

    return {
        "source_run_id": run_id,
        "source_run_date": run_date,
        "source_run_hour": run_hour,
        "source_run_timestamp_utc": run_timestamp,
    }


def as_gs_uri(path):
    return path if path.startswith("gs://") else f"gs://{path}"


# The leading wildcard also matches the older pmax_all_llm_outcomes.csv name.
discovered_paths = sorted(set(fs.glob(f"{RUNS_ROOT}/*/*all_llm_outcomes.csv")))

loaded_paths = {
    row.source_gcs_path
    for row in bq_client.query(
        f"select distinct source_gcs_path from `{TABLE_ID}`"
    ).result()
    if row.source_gcs_path
}

eligible_files = []
selected_files = []
for path in discovered_paths:
    run = parse_run(path)
    source_gcs_path = as_gs_uri(path)
    if run and run["source_run_timestamp_utc"] >= START_DATE:
        eligible_files.append((path, run))
        if source_gcs_path not in loaded_paths:
            selected_files.append((path, run))

print(f"Discovered {len(discovered_paths)} outcome files across all dates")
print(f"Eligible files at or after {START_DATE_TEXT}: {len(eligible_files)}")
print(f"Already loaded eligible files: {len(eligible_files) - len(selected_files)}")
print(f"New files selected for append: {len(selected_files)}")

if not selected_files:
    dbutils.notebook.exit("No new outcome files to load")

In [ ]:
frames = []
file_summary = []
empty_files = []

for path, run in selected_files:
    source_gcs_path = as_gs_uri(path)
    try:
        with fs.open(path, "rb") as file:
            frame = pd.read_csv(file, dtype=str)
    except pd.errors.EmptyDataError:
        empty_files.append(source_gcs_path)
        continue

    if frame.empty:
        empty_files.append(source_gcs_path)
        continue

    # Prefix metadata with source_ to avoid colliding with outcome columns.
    frame["source_run_id"] = run["source_run_id"]
    frame["source_run_date"] = run["source_run_date"]
    frame["source_run_hour"] = run["source_run_hour"]
    frame["source_run_timestamp_utc"] = run["source_run_timestamp_utc"]
    frame["source_gcs_path"] = source_gcs_path
    frame["source_row_number"] = range(1, len(frame) + 1)
    frame["source_record_id"] = (
        frame["source_gcs_path"]
        + "#"
        + frame["source_row_number"].astype(str)
    )

    frames.append(frame)
    file_summary.append(
        {
            "source_run_id": run["source_run_id"],
            "source_run_timestamp_utc": run["source_run_timestamp_utc"],
            "rows": len(frame),
            "source_gcs_path": source_gcs_path,
        }
    )

if empty_files:
    print(f"Skipped {len(empty_files)} empty outcome files")

if not frames:
    dbutils.notebook.exit("New outcome files were empty; no rows to load")

combined_df = pd.concat(frames, ignore_index=True, sort=False)

metadata_columns = [
    "source_record_id",
    "source_run_id",
    "source_run_date",
    "source_run_hour",
    "source_run_timestamp_utc",
    "source_gcs_path",
    "source_row_number",
]
outcome_columns = [
    column for column in combined_df.columns if column not in metadata_columns
]
combined_df = combined_df[metadata_columns + outcome_columns]
combined_df = combined_df.sort_values(
    ["source_run_timestamp_utc", "source_gcs_path", "source_row_number"]
).reset_index(drop=True)

print(f"Prepared {len(combined_df):,} rows from {len(file_summary)} new files")
display(pd.DataFrame(file_summary))

In [ ]:
table_columns = [field.name for field in TABLE_SCHEMA]
string_columns = [
    field.name for field in TABLE_SCHEMA if field.field_type == "STRING"
]
integer_columns = [
    field.name for field in TABLE_SCHEMA if field.field_type == "INTEGER"
]

for column in table_columns:
    if column not in combined_df.columns:
        combined_df[column] = pd.NA

combined_df["source_run_date"] = pd.to_datetime(
    combined_df["source_run_date"], errors="raise", utc=True
).dt.date
combined_df["source_run_timestamp_utc"] = pd.to_datetime(
    combined_df["source_run_timestamp_utc"], errors="raise", utc=True
)
combined_df["loaded_at_utc"] = pd.Timestamp.now(tz="UTC")

for column in integer_columns:
    numeric_values = pd.to_numeric(
        combined_df[column].astype("string").str.replace(",", "", regex=False),
        errors="raise",
    )
    if column == "confidence":
        numeric_values = numeric_values.round()
    combined_df[column] = numeric_values.astype("Int64")

boolean_values = (
    combined_df["scrape_failed"].astype("string").str.strip().str.lower()
)
boolean_map = {
    "true": True,
    "1": True,
    "yes": True,
    "false": False,
    "0": False,
    "no": False,
}
invalid_boolean_values = boolean_values[
    boolean_values.notna() & ~boolean_values.isin(boolean_map)
].unique()
if len(invalid_boolean_values):
    raise ValueError(
        "Unexpected scrape_failed values: "
        + ", ".join(sorted(invalid_boolean_values))
    )
combined_df["scrape_failed"] = boolean_values.map(boolean_map).astype("boolean")

for column in string_columns:
    combined_df[column] = combined_df[column].astype("string")

combined_df = combined_df[table_columns]

if combined_df["source_record_id"].duplicated().any():
    raise RuntimeError("Duplicate source_record_id values found in the new rows")

count_query = f"""
select
  count(*) as row_count,
  count(distinct source_record_id) as distinct_record_count
from `{TABLE_ID}`
"""

before_stats = next(iter(bq_client.query(count_query).result()))

job_config = bigquery.LoadJobConfig(
    schema=TABLE_SCHEMA,
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)
load_job = bq_client.load_table_from_dataframe(
    combined_df,
    TABLE_ID,
    job_config=job_config,
)
load_job.result()

after_stats = next(iter(bq_client.query(count_query).result()))
expected_row_count = before_stats.row_count + len(combined_df)

if after_stats.row_count != expected_row_count:
    raise RuntimeError(
        f"Append verification failed: expected {expected_row_count:,} total rows "
        f"but found {after_stats.row_count:,}"
    )

if after_stats.row_count != after_stats.distinct_record_count:
    raise RuntimeError(
        "Duplicate source_record_id values exist in the destination table"
    )

print(f"Appended {len(combined_df):,} rows from {len(file_summary)} files")
print(f"Verified {after_stats.row_count:,} total rows in {TABLE_ID}")
print(
    "Appended run range: "
    f"{combined_df['source_run_timestamp_utc'].min()} through "
    f"{combined_df['source_run_timestamp_utc'].max()}"
)

display(combined_df)